In [1]:
import json
import random
import uuid
from datetime import datetime, UTC

from faker import Faker
from kafka import KafkaProducer
from kafka.admin import KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import KafkaError
from kafka.errors import TopicAlreadyExistsError

In [2]:
def get_admin(kafka_config):
    return KafkaAdminClient(**kafka_config)


def list_topics(kafka_admin_client):
    topics = kafka_admin_client.list_topics()
    for t in sorted(topics):
        print(t)
    kafka_admin_client.close()


def create_topic(admin: KafkaAdminClient, topic_name: str, partitions: int = 3, replication_factor: int = 3, timeout_ms: int = 30000):
    new_topics = [NewTopic(name=topic_name, num_partitions=partitions, replication_factor=replication_factor)]
    try:
        resp = admin.create_topics(new_topics=new_topics, timeout_ms=timeout_ms)
        print(f"created topic: {topic_name}")
        return resp
    except TopicAlreadyExistsError:
        print(f"topic already exists: {topic_name}")
    except KafkaError as e:
        raise RuntimeError(f"create_topics failed for {topic_name}: {e}") from e


def delete_topics(kafka_admin_client, topics_to_delete):
    if isinstance(topics_to_delete, str):
        topics_to_delete = [topics_to_delete]
    print("Deleting topics:", topics_to_delete)
    kafka_admin_client.delete_topics(topics=topics_to_delete, timeout_ms=30000)
    kafka_admin_client.close()


def get_producer(kafka_config):
    return KafkaProducer(
        **kafka_config,
        key_serializer=lambda k: k.encode("utf-8") if k else None,
        value_serializer=lambda v: json.dumps(v, ensure_ascii=False).encode("utf-8"),
        acks="all",
        retries=1,
        request_timeout_ms=1500,
        api_version_auto_timeout_ms=1000,
        connections_max_idle_ms=2000)


def generate_product(fake_: Faker) -> dict:
    return {
        "product_id": str(uuid.uuid4()),
        "name": fake_.catch_phrase(),
        "category": random.choice(["전자기기", "생활용품", "식품", "도서", "의류", "스포츠", "뷰티"]),
        "price": random.randint(1_000, 500_000),
        "currency": "KRW",
        "stock": random.randint(0, 500),
        "created_at": datetime.now(UTC).isoformat()
    }

In [6]:
_bootstrap_servers = ("kafka-broker-1:9093,kafka-broker-2:9096,kafka-broker-3:9099")
_kafka_config = {
    "bootstrap_servers": _bootstrap_servers,
    "security_protocol": "SASL_PLAINTEXT",
    "sasl_mechanism": "PLAIN",
    "sasl_plain_username": "admin",
    "sasl_plain_password": "admin"
}

In [7]:
#delete_topics(get_admin(_kafka_config), "mmix-product-topic")
create_topic(get_admin(_kafka_config), "mmix-product-topic")
list_topics(get_admin(_kafka_config))

topic already exists: mmix-product-topic
__consumer_offsets
debezium-connect-configs
debezium-connect-offsets
debezium-connect-status
debezium.mysql.source
debezium.mysql.source.mmix.departments
debezium.mysql.source.mmix.employees
debezium.mysql.source.mmix.etl_analysis_logs
debezium.mysql.source.mmix.news_articles
debezium.mysql.source.schema.changes.inventory
mmix-product-topic


In [8]:
import time

producer = get_producer(_kafka_config)

try:
    for i in range(100):
        producer.send(topic="mmix-product-topic", key=f"key-{i}", value=generate_product(Faker(locale="ko_KR")))
        time.sleep(1)
    producer.flush()
finally:
    producer.close()